In [1]:
import cv2
import numpy as np

In [11]:
image = cv2.imread('pic.jpg')
cv2.imshow('Original', image)
cv2.waitKey(0)
cv2.destroyAllWindows()

In [12]:
gray_image = cv2.cvtColor(image,cv2.COLOR_BGR2GRAY)
img = cv2.resize(gray_image,(512,512))
cv2.imshow('Resized',img)
cv2.waitKey(0)
cv2.destroyAllWindows()

In [4]:
height, width = img.shape
min_dimension = min(height, width)
scale_height = height / 256
scale_width = width / 256
square_matrix = np.zeros((256, 256), dtype=np.uint8)

In [13]:
for i in range(256):
    for j in range(256):
        orig_x = int(j * scale_width)
        orig_y = int(i * scale_height)
        orig_x = min(orig_x, width - 1)
        orig_y = min(orig_y, height - 1)
        square_matrix[i, j] = img[orig_y, orig_x]


In [14]:
cv2.imwrite('img.jpg', square_matrix)
print(f"Resized Matrix Shape: {square_matrix.shape}")


Resized Matrix Shape: (256, 256)


In [15]:
def generate_hadamard_matrix(n):
    if n == 1:
        return np.array([[1]]) 
    H_n_minus_1 = generate_hadamard_matrix(n // 2)
    top = np.hstack((H_n_minus_1, H_n_minus_1))
    bottom = np.hstack((H_n_minus_1, -H_n_minus_1))  
    return np.vstack((top, bottom))

def hadamard_transform(matrix):
    n = matrix.shape[0]
    H = generate_hadamard_matrix(n)
    transformed_matrix = np.dot(H, np.dot(matrix, H))  
    return transformed_matrix

def hadamard_inverse(matrix):
    n = matrix.shape[0]
    H = generate_hadamard_matrix(n)
    H_inv = (1 / n) * H.T
    return np.dot(H_inv, np.dot(matrix, H_inv))

In [16]:
size = 256 
resized_image = cv2.resize(img, (size, size))
transformed_image = hadamard_transform(resized_image)
inverse_transformed_image = hadamard_inverse(transformed_image)

cv2.imshow("Original Image", resized_image)
cv2.imshow("Hadamard Transformed Image", (transformed_image).astype(np.uint8))
cv2.imshow("Inverse Transformed Image", inverse_transformed_image.astype(np.uint8))

cv2.waitKey(0)
cv2.destroyAllWindows()


In [17]:
def custom_hstack(A, B):
    n, m = A.shape
    result = np.zeros((n, m * 2))
    for i in range(n):
        for j in range(m):
            result[i, j] = A[i, j]
            result[i, j + m] = B[i, j]
    return result

def custom_vstack(A, B):
    n, m = A.shape
    result = np.zeros((n * 2, m))
    for i in range(n):
        for j in range(m):
            result[i, j] = A[i, j]
            result[i + n, j] = B[i, j]
    return result

def custom_matrix_mult(A, B):
    n = A.shape[0]
    result = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            for k in range(n):
                result[i, j] += A[i, k] * B[k, j]
    return result

def custom_scalar_mult(A, scalar):
    n, m = A.shape
    result = np.zeros((n, m))
    for i in range(n):
        for j in range(m):
            result[i, j] = A[i, j] * scalar
    return result

def eliminate_quadrants(hadamard_image, percentage):
    h, w = hadamard_image.shape
    mid_h = h // 2
    mid_w = w // 2
    modified_image = hadamard_image.copy()
    if percentage == 25:
        modified_image[mid_h:, mid_w:] = 0
    elif percentage == 50:
        modified_image[mid_h:, :] = 0
    elif percentage == 75:
        modified_image[mid_h:, :] = 0   
        modified_image[:mid_h, mid_w:] = 0
    return modified_image


In [18]:
size = 256
resized_image = cv2.resize(img, (size, size))

transformed_image = hadamard_transform(resized_image)
log_transformed_image = np.log1p(np.abs(transformed_image))

hadamard_25 = eliminate_quadrants(transformed_image, 25)
hadamard_50 = eliminate_quadrants(transformed_image, 50)
hadamard_75 = eliminate_quadrants(transformed_image, 75)


In [19]:
cv2.imshow("Original Image", resized_image)
cv2.imshow("Hadamard Transformed Image", log_transformed_image.astype(np.uint8))
cv2.imshow("25% Frequencies Removed",np.abs(hadamard_25).astype(np.uint8))
cv2.imshow("50% Frequencies Removed", np.abs(hadamard_50).astype(np.uint8))
cv2.imshow("75% Frequencies Removed", np.abs(hadamard_75).astype(np.uint8))
cv2.waitKey(0)
cv2.destroyAllWindows()



In [13]:
reconstructed_25 = hadamard_inverse(hadamard_25)
reconstructed_50 = hadamard_inverse(hadamard_50)
reconstructed_75 = hadamard_inverse(hadamard_75)

cv2.imshow("Reconstructed 25%", reconstructed_25.astype(np.uint8))
cv2.imshow("Reconstructed 50%", reconstructed_50.astype(np.uint8))
cv2.imshow("Reconstructed 75%", reconstructed_75.astype(np.uint8))

cv2.waitKey(0)
cv2.destroyAllWindows()

In [20]:
def dct_1d(x):
    N = len(x)
    X = np.zeros(N)
    for k in range(N):
        alpha_k = np.sqrt(1 / N) if k == 0 else np.sqrt(2 / N)
        X[k] = alpha_k * np.sum(x * np.cos(np.pi * (2 * np.arange(N) + 1) * k / (2 * N)))
    return X

def idct_1d(X):
    N = len(X)
    x = np.zeros(N)
    for n in range(N):
        x[n] = np.sum(X * np.cos(np.pi * (2 * np.arange(N) + 1) * n / (2 * N)))
    x = x * np.sqrt(2 / N)
    x[0] = x[0] / np.sqrt(2)
    return x

def dct_2d(matrix):
    M, N = matrix.shape
    dct_rows = np.apply_along_axis(dct_1d, 1, matrix)
    dct_matrix = np.apply_along_axis(dct_1d, 0, dct_rows)
    return dct_matrix

def idct_2d(matrix):
    M, N = matrix.shape
    idct_cols = np.apply_along_axis(idct_1d, 0, matrix)
    idct_matrix = np.apply_along_axis(idct_1d, 1, idct_cols)
    return idct_matrix



In [21]:
size = 256
resized_image = cv2.resize(img, (size, size))

transformed_image = hadamard_transform(resized_image)
inverse_transformed_image = hadamard_inverse(transformed_image)
dct_image = dct_2d(resized_image)

cv2.imshow("Original Image", resized_image)
cv2.imshow("Hadamard Transformed Image", np.abs(transformed_image).astype(np.uint8))
cv2.imshow("Inverse Hadamard Image", inverse_transformed_image.astype(np.uint8))
cv2.imshow("Full DCT", np.abs(dct_image).astype(np.uint8))

cv2.waitKey(0)
cv2.destroyAllWindows()

In [22]:
def apply_frequency_elimination(dct_image, percentage):
    M, N = dct_image.shape
    num_coeffs_to_keep = int((percentage / 100) * M * N)

    dct_image_flat = dct_image.flatten()
    abs_dct_image_flat = np.abs(dct_image_flat)
    sorted_indices = np.argsort(abs_dct_image_flat)[::-1]

    threshold_index = sorted_indices[num_coeffs_to_keep]
    threshold_value = abs_dct_image_flat[threshold_index]

    mask = np.abs(dct_image) >= threshold_value
    dct_image_filtered = dct_image * mask
    return dct_image_filtered


In [23]:
size = 256
resized_image = cv2.resize(img, (size, size))

dct_image = dct_2d(resized_image)

dct_image_25 = apply_frequency_elimination(dct_image, 25)
dct_image_50 = apply_frequency_elimination(dct_image, 50)
dct_image_75 = apply_frequency_elimination(dct_image, 75)

idct_image_25 = idct_2d(dct_image_25)
idct_image_50 = idct_2d(dct_image_50)
idct_image_75 = idct_2d(dct_image_75)

cv2.imshow("Original Image", resized_image)
cv2.imshow("Full DCT", np.abs(dct_image).astype(np.uint8))
cv2.imshow("DCT - 25% Retained", np.abs(dct_image_25).astype(np.uint8))
cv2.imshow("IDCT - 25% Retained", idct_image_25.astype(np.uint8))
cv2.imshow("DCT - 50% Retained", np.abs(dct_image_50).astype(np.uint8))
cv2.imshow("IDCT - 50% Retained", idct_image_50.astype(np.uint8))
cv2.imshow("DCT - 75% Retained", np.abs(dct_image_75).astype(np.uint8))
cv2.imshow("IDCT - 75% Retained", idct_image_75.astype(np.uint8))

cv2.waitKey(0)
cv2.destroyAllWindows()


In [24]:
def block_dct_2d(image, block_size):
    M, N = image.shape
    dct_image = np.zeros((M, N))
    for i in range(0, M, block_size):
        for j in range(0, N, block_size):
            block = image[i:i+block_size, j:j+block_size]
            dct_block = dct_2d(block)
            dct_image[i:i+block_size, j:j+block_size] = dct_block
    return dct_image

def block_idct_2d(image, block_size):
    M, N = image.shape
    idct_image = np.zeros((M, N))
    for i in range(0, M, block_size):
        for j in range(0, N, block_size):
            block = image[i:i+block_size, j:j+block_size]
            idct_block = idct_2d(block)
            idct_image[i:i+block_size, j:j+block_size] = idct_block
    return idct_image



In [25]:
size = 256
resized_image = cv2.resize(img, (size, size))

transformed_image = hadamard_transform(resized_image)
inverse_transformed_image = hadamard_inverse(transformed_image)

dct_image_4x4 = block_dct_2d(resized_image, 4)
idct_image_4x4 = block_idct_2d(dct_image_4x4, 4)
dct_image_8x8 = block_dct_2d(resized_image, 8)
idct_image_8x8 = block_idct_2d(dct_image_8x8, 8)

cv2.imshow("Original Image", resized_image)
cv2.imshow("Hadamard Transformed Image", np.abs(transformed_image).astype(np.uint8))
cv2.imshow("Inverse Hadamard Image", inverse_transformed_image.astype(np.uint8))
cv2.imshow("DCT Image (4x4 Blocks)", np.abs(dct_image_4x4).astype(np.uint8))
cv2.imshow("IDCT Image (4x4 Blocks)", idct_image_4x4.astype(np.uint8))
cv2.imshow("DCT Image (8x8 Blocks)", np.abs(dct_image_8x8).astype(np.uint8))
cv2.imshow("IDCT Image (8x8 Blocks)", idct_image_8x8.astype(np.uint8))

cv2.waitKey(0)
cv2.destroyAllWindows()


In [26]:
def apply_frequency_elimination(dct_image, percentage):
    M, N = dct_image.shape
    num_coeffs_to_keep = int((percentage / 100) * M * N)
    dct_image_flat = dct_image.flatten()
    abs_dct_image_flat = np.abs(dct_image_flat)
    sorted_indices = np.argsort(abs_dct_image_flat)[::-1]
    threshold_index = sorted_indices[num_coeffs_to_keep]
    threshold_value = abs_dct_image_flat[threshold_index]
    mask = np.abs(dct_image) >= threshold_value
    
    dct_image_filtered = dct_image * mask
    return dct_image_filtered




In [27]:
size = 256
resized_image = cv2.resize(img, (size, size))

dct_image_8x8 = block_dct_2d(resized_image, 8)

dct_image_8x8_25 = apply_frequency_elimination(dct_image_8x8, 25)
dct_image_8x8_50 = apply_frequency_elimination(dct_image_8x8, 50)
dct_image_8x8_75 = apply_frequency_elimination(dct_image_8x8, 75)

idct_image_8x8_25 = block_idct_2d(dct_image_8x8_25, 8)
idct_image_8x8_50 = block_idct_2d(dct_image_8x8_50, 8)
idct_image_8x8_75 = block_idct_2d(dct_image_8x8_75, 8)

cv2.imshow("Original Image", resized_image)
cv2.imshow("DCT Image (8x8 Blocks) - 25% Retained", np.abs(dct_image_8x8_25).astype(np.uint8))
cv2.imshow("IDCT Image (8x8 Blocks) - 25% Retained", idct_image_8x8_25.astype(np.uint8))
cv2.imshow("DCT Image (8x8 Blocks) - 50% Retained", np.abs(dct_image_8x8_50).astype(np.uint8))
cv2.imshow("IDCT Image (8x8 Blocks) - 50% Retained", idct_image_8x8_50.astype(np.uint8))
cv2.imshow("DCT Image (8x8 Blocks) - 75% Retained", np.abs(dct_image_8x8_75).astype(np.uint8))
cv2.imshow("IDCT Image (8x8 Blocks) - 75% Retained", idct_image_8x8_75.astype(np.uint8))

cv2.waitKey(0)
cv2.destroyAllWindows()

In [2]:
animal_image = cv2.imread('animal.jpg')
cv2.imshow('Original animal', animal_image)
cv2.waitKey(0)
cv2.destroyAllWindows()

In [28]:
animal_gray_image = cv2.cvtColor(animal_image,cv2.COLOR_BGR2GRAY)
img = cv2.resize(animal_gray_image,(512,512))
cv2.imshow('Resized animal',img)
cv2.waitKey(0)
cv2.destroyAllWindows()

In [5]:
height, width = img.shape
min_dimension = min(height, width)
scale_height = height / 256
scale_width = width / 256
square_matrix = np.zeros((256, 256), dtype=np.uint8)

In [29]:
for i in range(256):
    for j in range(256):
        orig_x = int(j * scale_width)
        orig_y = int(i * scale_height)
        orig_x = min(orig_x, width - 1)
        orig_y = min(orig_y, height - 1)
        square_matrix[i, j] = img[orig_y, orig_x]

In [30]:
cv2.imwrite('img.jpg', square_matrix)
print(f"Resized Matrix Shape: {square_matrix.shape}")

Resized Matrix Shape: (256, 256)


In [33]:
def generate_hadamard_matrix(n):
    if n == 1:
        return np.array([[1]])
    H_n_minus_1 = generate_hadamard_matrix(n // 2)
    top = np.hstack((H_n_minus_1, H_n_minus_1))
    bottom = np.hstack((H_n_minus_1, -H_n_minus_1))
    return np.vstack((top, bottom))

def hadamard_transform(matrix):
    n = matrix.shape[0]
    H = generate_hadamard_matrix(n)
    transformed_matrix = np.dot(H, np.dot(matrix, H))
    return transformed_matrix

def hadamard_inverse(matrix):
    n = matrix.shape[0]
    H = generate_hadamard_matrix(n)
    H_inv = (1 / n) * H.T
    return np.dot(H_inv, np.dot(matrix, H_inv))


In [34]:
size = 256
resized_image = cv2.resize(img, (size, size))

transformed_image = hadamard_transform(resized_image)

inverse_transformed_image = hadamard_inverse(transformed_image)

log_transformed = np.log1p(np.abs(transformed_image))

cv2.normalize(log_transformed, log_transformed, 0, 255, cv2.NORM_MINMAX)
log_transformed = np.uint8(log_transformed)

cv2.imshow("Original Image", resized_image)
cv2.imshow("Hadamard Transformed Image", log_transformed)
cv2.imshow("Inverse Transformed Image", np.uint8(inverse_transformed_image))

cv2.waitKey(0)
cv2.destroyAllWindows()


In [35]:
def generate_hadamard_matrix(n):
    if n == 1:
        return np.array([[1]])
    H_n_minus_1 = generate_hadamard_matrix(n // 2)
    top = custom_hstack(H_n_minus_1, H_n_minus_1)
    bottom = custom_hstack(H_n_minus_1, custom_scalar_mult(H_n_minus_1, -1))
    return custom_vstack(top, bottom)

def custom_hstack(A, B):
    n, m = A.shape
    result = np.zeros((n, m * 2))
    for i in range(n):
        for j in range(m):
            result[i, j] = A[i, j]
            result[i, j + m] = B[i, j]
    return result

def custom_vstack(A, B):
    n, m = A.shape
    result = np.zeros((n * 2, m))
    for i in range(n):
        for j in range(m):
            result[i, j] = A[i, j]
            result[i + n, j] = B[i, j]
    return result

def custom_matrix_mult(A, B):
    n = A.shape[0]
    result = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            for k in range(n):
                result[i, j] += A[i, k] * B[k, j]
    return result

def custom_scalar_mult(A, scalar):
    n, m = A.shape
    result = np.zeros((n, m))
    for i in range(n):
        for j in range(m):
            result[i, j] = A[i, j] * scalar
    return result

# Hadamard Transform
def hadamard_transform(matrix):
    n = matrix.shape[0]
    H = generate_hadamard_matrix(n)
    transformed_matrix = custom_matrix_mult(H, custom_matrix_mult(matrix, H))
    return transformed_matrix

def hadamard_inverse(matrix):
    n = matrix.shape[0]
    H = generate_hadamard_matrix(n)
    H_inv = custom_scalar_mult(H.T, 1 / n)
    return custom_matrix_mult(H_inv, custom_matrix_mult(matrix, H_inv))

def eliminate_quadrants(hadamard_image, percentage):
    h, w = hadamard_image.shape
    mid_h = h // 2
    mid_w = w // 2
    modified_image = hadamard_image.copy()
    if percentage == 25:
        modified_image[mid_h:, mid_w:] = 0
    elif percentage == 50:
        modified_image[mid_h:, :] = 0
    elif percentage == 75:
        modified_image[mid_h:, :] = 0
        modified_image[:mid_h, mid_w:] = 0
    return modified_image


In [41]:
size = 256
resized_image = cv2.resize(img, (size, size))

transformed_image = hadamard_transform(resized_image)

# Apply logarithmic transformation for better visualization
log_transformed_image = np.abs(transformed_image)

# Eliminate frequencies
hadamard_25 = eliminate_quadrants(transformed_image, 25)
hadamard_50 = eliminate_quadrants(transformed_image, 50)
hadamard_75 = eliminate_quadrants(transformed_image, 75)

# Normalize the images for proper display
cv2.normalize(log_transformed_image, log_transformed_image, 0, 255, cv2.NORM_MINMAX)
log_transformed_image = np.uint8(log_transformed_image)

cv2.normalize(hadamard_25, hadamard_25, 0, 255, cv2.NORM_MINMAX)
cv2.normalize(hadamard_50, hadamard_50, 0, 255, cv2.NORM_MINMAX)
cv2.normalize(hadamard_75, hadamard_75, 0, 255, cv2.NORM_MINMAX)

# Reconstructed images after eliminating frequencies
reconstructed_25 = hadamard_inverse(hadamard_25)
reconstructed_50 = hadamard_inverse(hadamard_50)
reconstructed_75 = hadamard_inverse(hadamard_75)

# Display images
cv2.imshow("Original Image", resized_image)
cv2.imshow("Hadamard Transformed Image", log_transformed_image)
cv2.imshow("25% Frequencies Removed", np.uint8(np.abs(hadamard_25)))
cv2.imshow("50% Frequencies Removed", np.uint8(np.abs(hadamard_50)))
cv2.imshow("75% Frequencies Removed", np.uint8(np.abs(hadamard_75)))
cv2.imshow("Reconstructed 25%", np.uint8(reconstructed_25))
cv2.imshow("Reconstructed 50%", np.uint8(reconstructed_50))
cv2.imshow("Reconstructed 75%", np.uint8(reconstructed_75))

# Wait for a key press and close all windows
cv2.waitKey(0)
cv2.destroyAllWindows()



In [7]:
def generate_hadamard_matrix(size):
    H = np.array([[1]])
    while H.shape[0] < size:
        H_half = H
        top_left = H_half
        top_right = H_half
        bottom_left = H_half
        bottom_right = -H_half
        top = np.hstack((top_left, top_right))
        bottom = np.hstack((bottom_left, bottom_right))
        H = np.vstack((top, bottom))
    return H

In [8]:
def hadamard_transform(f, H):
     H_T = H.T
     return np.dot(H, np.dot(f, H_T))

In [11]:
def inverse_hadamard_transform(F, H):
     H_T = H.T
     return np.dot(H_T, np.dot(F, H)) / (H.shape[0] ** 2)

In [13]:
H = generate_hadamard_matrix(512)
 
F = hadamard_transform(img, H)
 
reconstructed_img = inverse_hadamard_transform(F, H)
 
cv2.imshow('Original Image', img)
cv2.imshow('Transformed Image', np.clip(F, 0, 255).astype(np.uint8))  # Clip values for display
cv2.imshow('Reconstructed Image', np.clip(reconstructed_img, 0, 255).astype(np.uint8))  # Clip values for display
 
cv2.waitKey(0)
cv2.destroyAllWindows()

In [14]:
new = img - reconstructed_img

In [15]:
cv2.imshow('Subtracted ', new)
cv2.waitKey(0)
cv2.destroyAllWindows()